# Signal-to-Noise Ratio Quality Analysis

This notebook analyzes the Signal-to-Noise Ratio (SNR) of voice recordings and helps identify files that should be cleaned up or removed before training.

**SNR in context:** SNR is a technical quality metric, but the decision to include or exclude a recording is a community decision. A low-SNR recording of an elder speaking irreplaceable language may be worth cleaning or including despite noise. A high-SNR recording of generic content may be less valuable. SNR is a quality gate, not the only gate.

This notebook was migrated from the original Google Colab version. It now runs locally — no internet connection required, and no audio leaves your machine.

**As a rough guide:** SNR above 20 dB is excellent; 15–20 dB is acceptable; below 15 dB warrants review.


## 1. Setup

This notebook requires `ffmpeg` and the WADA_SNR tool (a C program from CMU). Run the cells below to build it.

```bash
uv sync --extra dev
```

In [ ]:
import subprocess, sys, os

# Check for ffmpeg
result = subprocess.run(["ffmpeg", "-version"], capture_output=True)
if result.returncode != 0:
    print("ffmpeg not found. Install it: https://ffmpeg.org/download.html")
else:
    print("ffmpeg OK")

In [ ]:
import os

WADA_DIR = os.path.abspath("../tools/WADA_SNR")
WADA_BIN = os.path.join(WADA_DIR, "bin", "WADASNR")

if os.path.exists(WADA_BIN):
    print("WADA_SNR already built.")
elif os.path.exists(os.path.join(WADA_DIR, "WADASNR")):
    # Binary already compiled in extract directory — just stage it
    os.makedirs(os.path.join(WADA_DIR, "bin"), exist_ok=True)
    subprocess.run(f"cp {WADA_DIR}/WADASNR {WADA_BIN}", shell=True, check=True)
    print("WADA_SNR staged to bin/.")
else:
    # Fresh download and build
    os.makedirs(WADA_DIR, exist_ok=True)
    subprocess.run(
        "wget -q http://www.cs.cmu.edu/~robust/archive/algorithms/"
        "WADA_SNR_IS_2008/WadaSNR.tar.gz -O /tmp/wada_snr.tar.gz",
        shell=True, check=True
    )
    subprocess.run(
        f"tar -xzf /tmp/wada_snr.tar.gz -C {WADA_DIR} --strip-components=1",
        shell=True, check=True
    )
    subprocess.run("make", cwd=WADA_DIR, check=True)
    os.makedirs(os.path.join(WADA_DIR, "bin"), exist_ok=True)
    subprocess.run(f"cp {WADA_DIR}/WADASNR {WADA_BIN}", shell=True, check=True)
    print("WADA_SNR built.")

print(f"Binary: {WADA_BIN}")

## 2. Configure

In [ ]:
WAVS_DIR = "../test_data/wavs_export_audacity/"
TEMP_DIR = "/tmp/snr_16k/"  # Temporary 16kHz versions for WADA_SNR
os.makedirs(TEMP_DIR, exist_ok=True)

## 3. Compute SNR

In [ ]:
import soundfile as sf
import pandas as pd
import re

TABLE_FILE = os.path.join(WADA_DIR, "Alpha0.400000.txt")  # standard alpha=0.4 lookup table

def compute_snr(wav_path: str, wada_bin: str, wada_dir: str, table_file: str, temp_dir: str) -> float | None:
    """Convert to 16kHz and compute SNR using WADA_SNR."""
    basename = os.path.basename(wav_path)
    tmp_path = os.path.join(temp_dir, basename)

    subprocess.run(
        f'ffmpeg -y -i "{wav_path}" -ar 16000 -ac 1 "{tmp_path}" -loglevel quiet',
        shell=True, check=True
    )

    result = subprocess.run(
        [wada_bin, "-i", tmp_path, "-ifmt", "mswav", "-t", table_file],
        capture_output=True,
        cwd=wada_dir
    )
    output = (result.stdout + result.stderr).decode("latin-1", errors="replace")

    # Match: "Total SNR is 14.000000 dB."
    for line in output.splitlines():
        m = re.search(r"Total SNR is\s+([\d.]+)\s+dB", line)
        if m:
            return float(m.group(1))
    return None


wav_files = sorted(f for f in os.listdir(WAVS_DIR) if f.endswith(".wav"))
print(f"Analyzing {len(wav_files)} files...")

rows = []
for fname in wav_files:
    path = os.path.join(WAVS_DIR, fname)
    snr = compute_snr(path, WADA_BIN, WADA_DIR, TABLE_FILE, TEMP_DIR)
    rows.append({"file": fname, "snr_db": snr})
    print(f"  {fname}: {snr} dB")

df = pd.DataFrame(rows).sort_values("snr_db")
print("\nResults (sorted by SNR):")
print(df.to_string(index=False))

## 4. Visualize Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["snr_db"].dropna(), bins=20, edgecolor="black")
ax.axvline(15, color="orange", linestyle="--", label="15 dB (review threshold)")
ax.axvline(20, color="green", linestyle="--", label="20 dB (good quality)")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Number of files")
ax.set_title("Signal-to-Noise Ratio Distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFiles below 15 dB (review recommended): {len(df[df['snr_db'] < 15])}")
print(f"Files above 20 dB (good quality): {len(df[df['snr_db'] >= 20])}")

## 5. Listen to Files by Quality

Play files from worst to best SNR to calibrate your quality threshold.

In [ ]:
from IPython.display import Audio, display

for _, row in df.iterrows():
    fpath = os.path.join(WAVS_DIR, row["file"])
    print(f"{row['file']} — {row['snr_db']} dB")
    display(Audio(fpath))

## 6. Flag Low-Quality Files in Metadata

Files below your threshold should have `exclude_from_training = true` and `exclude_reason = "low SNR quality"` in `metadata/metadata_template.csv`.

Remember: the decision to exclude is yours (and the community's). A noisy recording of irreplaceable content may still belong in the archive even if it is excluded from training.

In [ ]:
SNR_THRESHOLD = 15  # dB — adjust to your dataset

low_quality = df[df["snr_db"] < SNR_THRESHOLD]
if len(low_quality) > 0:
    print(f"Files recommended for exclusion from training (SNR < {SNR_THRESHOLD} dB):")
    for _, row in low_quality.iterrows():
        file_id = os.path.splitext(row["file"])[0]
        print(f"  file_id={file_id}, snr={row['snr_db']} dB")
    print("\nSet exclude_from_training=true and exclude_reason='low SNR quality' in metadata_template.csv")
else:
    print(f"All files are above {SNR_THRESHOLD} dB threshold.")

## Next Steps — Choose Your Transcription Path

| Your language | Notebook |
|---|---|
| English, Spanish, Māori, or other Whisper-supported language | [03a_transcribe_whisper.ipynb](03a_transcribe_whisper.ipynb) |
| Navajo or other language covered by Meta MMS | [03b_transcribe_mms.ipynb](03b_transcribe_mms.ipynb) |
| Language not covered by either tool, or community prefers manual | [03c_transcribe_manual.ipynb](03c_transcribe_manual.ipynb) |

Not sure? Check the [Whisper language list](https://github.com/openai/whisper#available-models-and-languages) and [MMS language list](https://dl.fbaipublicfiles.com/mms/misc/language_coverage_mms.html).